# 02 — Role-Specific Performance Target Construction and Validation

**Input:** `data/processed/Dataset_PlayerSeason_Modelling.csv`  
**Output:** `data/processed/Role_Specific_Index_Dataset.csv`

This notebook constructs the dissertation's role-aware player-performance target before any predictive model is fitted. Outfield players are grouped into broad defender, midfielder and forward roles; four football-performance components are standardised within season and role; and a transparent role-specific composite is converted to a 0–100 percentile rank. The primary eligibility threshold is 450 minutes. A separate 900-minute target-level sensitivity reference is calculated to test whether tightening the target population materially changes player ordering, without introducing a second modelling exercise.

## 0. Setup

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.analysis_utils import paths
P = paths(ROOT)
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 20)

SEASON_ORDER = ['1718','1819','1920','2021','2122','2223','2324','2425']
COMPONENTS = ['attack','creation','possession','defense']
PRIMARY_W = {
    'FW': {'attack':0.50, 'creation':0.25, 'possession':0.15, 'defense':0.10},
    'MF': {'attack':0.20, 'creation':0.35, 'possession':0.25, 'defense':0.20},
    'DF': {'attack':0.10, 'creation':0.15, 'possession':0.25, 'defense':0.50},
}


## 1. Index Construction

### 1.1 Position groups and eligibility
Goalkeepers are excluded, consistent with the Research Design protocol's own recommendation. Outfield players are assigned a broad role (DF/MF/FW) from the first-listed FBref position code. A player-season is eligible if minutes played >= 450 (primary threshold); a 900-minute version is computed as the specified robustness check.

In [2]:
df = pd.read_csv(P["processed"] / "Dataset_PlayerSeason_Modelling.csv", dtype={'season': str})
df['season'] = pd.Categorical(df['season'], categories=SEASON_ORDER, ordered=True)

df['pos_primary'] = df['pos_'].astype(str).str.split(',').str[0]
df.loc[df['pos_primary']=='nan', 'pos_primary'] = np.nan
df['role'] = df['pos_primary'].map({'GK':'GK','DF':'DF','MF':'MF','FW':'FW'})

df['eligible_450'] = (df['Playing Time_Min'] >= 450) & (df['role'].isin(['DF','MF','FW']))
df['eligible_900'] = (df['Playing Time_Min'] >= 900) & (df['role'].isin(['DF','MF','FW']))

print("Eligible (450 min, outfield) player-seasons:", df['eligible_450'].sum(), "of", len(df))
print(df[df['eligible_450']]['role'].value_counts())


Eligible (450 min, outfield) player-seasons: 14661 of 21371
role
DF    6069
MF    5018
FW    3574
Name: count, dtype: int64


### 1.2 Component groups
Four per-90 component groups, each a sum of raw FBref counts divided by `90s_` (matches-equivalent played):

- **Attack** — non-penalty xG + xAG per 90 (already computed as `Per 90 Minutes_npxG+xAG` during the player-season aggregation).
- **Creation** — shot-creating actions + key passes + 0.5x(progressive passes + progressive carries), per 90.
- **Possession** — pass completion % (rescaled /20) + passes into the box + 0.5x passes into the final third + successful take-ons, per 90.
- **Defense** — tackles+interceptions + 0.5x(blocks + clearances + aerial duels won), per 90.

In [3]:
NINETY = df['90s_'].replace(0, np.nan)
def per90(col): return df[col] / NINETY

df['raw_attack'] = df['Per 90 Minutes_npxG+xAG']
df['raw_creation'] = (per90('SCA_SCA') + per90('KP_') + 0.5*per90('Progression_PrgP') + 0.5*per90('Progression_PrgC'))
df['raw_possession'] = (df['Total_Cmp%'].fillna(0)/20.0 + per90('PPA_') + 0.5*per90('1/3_') + per90('Take-Ons_Succ'))
df['raw_defense'] = (per90('Tkl+Int_') + 0.5*per90('Blocks_Blocks') + 0.5*per90('Clr_') + 0.5*per90('Aerial Duels_Won'))

df[['player','season','role','raw_attack','raw_creation','raw_possession','raw_defense']].dropna(subset=['role']).head()


,player,season,role,raw_attack,raw_creation,raw_possession,raw_defense
0,Aaron Ciammaglichella,2425,MF,NaN,NaN,NaN,NaN
1,Aaron Connolly,1920,FW,0.321429,3.464286,4.538018,1.642857
2,Aaron Connolly,2021,FW,0.477273,3.295455,4.682060,1.420455
3,Aaron Connolly,2122,FW,0.294118,2.941176,3.684211,1.176471
4,Aaron Cresswell,1718,DF,0.117302,5.322581,6.235841,5.454545


### 1.3 Standardisation
Each raw component is z-scored **within (season x role), pooled across all five leagues** — not within league. Standardising within league would force an identical role mean onto every league by construction, making "stability across leagues" (Section 3) untestable.

### 1.4 Role-specific weights and composite
Weights (Section 0) are grounded in the literature's role-aware evaluation frameworks (PlayeRank's per-role feature weighting; Evers et al.'s weighted sub-scores). Composite = sum(role weight x component z-score). A percentile version is also produced for interpretability.

In [4]:
def zscore_within(frame, cols, group_cols):
    g = frame.groupby(group_cols, observed=True)[cols]
    return (frame[cols] - g.transform('mean')) / g.transform('std')

def compute_index(df, weights, eligible_col='eligible_450'):
    elig = df[df[eligible_col] & df['role'].isin(['DF','MF','FW'])].copy()
    raw_cols = [f'raw_{c}' for c in COMPONENTS]
    z = zscore_within(elig, raw_cols, ['season','role'])
    z.columns = [f'z_{c}' for c in COMPONENTS]
    elig = pd.concat([elig, z], axis=1)
    idx = np.zeros(len(elig))
    for role, w in weights.items():
        mask = (elig['role']==role).values
        role_z = np.column_stack([elig.loc[mask, f'z_{c}'].values for c in COMPONENTS])
        role_w = np.array([w[c] for c in COMPONENTS])
        idx[mask] = role_z @ role_w
    elig['role_index_z'] = idx
    elig['role_index_pct'] = elig.groupby(['season','role'], observed=True)['role_index_z'].rank(pct=True) * 100
    elig['group_n'] = elig.groupby(['season','role'], observed=True)['role_index_z'].transform('size')
    return elig

primary = compute_index(df, PRIMARY_W, 'eligible_450')
primary900 = compute_index(df, PRIMARY_W, 'eligible_900')
print("Primary index (450 min):", len(primary), "player-seasons")
primary.groupby('role', observed=True)['role_index_z'].describe()


Primary index (450 min): 14661 player-seasons


,count,mean,std,min,25%,50%,75%,max
role,,,,,,,,
DF,6069.0,-8.078341e-17,0.514797,-1.992991,-0.352992,-0.032331,0.328140,2.216595
FW,3574.0,-4.771412e-17,0.670749,-1.566736,-0.465482,-0.100016,0.369085,4.132193
MF,5018.0,-1.274389e-17,0.622522,-1.423469,-0.426693,-0.082123,0.357362,3.973016


### 1.5 Target-level sensitivity at 900 minutes

The 900-minute reference is used only to test the stability of the target definition under a stricter playing-time threshold. For player-seasons eligible under both thresholds, the 450-minute percentile rank (defined relative to the broader 450-minute population) is compared with the independently recomputed 900-minute percentile rank. This is a target-construction sensitivity check, not a second model evaluation.

In [5]:
target_900_compare = primary[['player_id','season','role','role_index_pct']].rename(columns={'role_index_pct':'pctile_450_reference'}).merge(
    primary900[['player_id','season','role_index_pct']].rename(columns={'role_index_pct':'pctile_900_reference'}),
    on=['player_id','season'], how='inner'
)
target_900_rows=[]
for label,g in [('Overall',target_900_compare), *[(r,target_900_compare[target_900_compare['role']==r]) for r in ['DF','MF','FW']]]:
    rho=stats.spearmanr(g['pctile_450_reference'],g['pctile_900_reference']).statistic
    diff=(g['pctile_450_reference']-g['pctile_900_reference']).abs()
    target_900_rows.append({
        'group':label,'n':len(g),'spearman_rho':rho,
        'mean_absolute_percentile_difference':diff.mean(),
        'median_absolute_percentile_difference':diff.median(),
        'p95_absolute_percentile_difference':diff.quantile(.95),
    })
target_900_sensitivity=pd.DataFrame(target_900_rows)
display(target_900_sensitivity.round(4))

,group,n,spearman_rho,mean_absolute_percentile_difference,median_absolute_percentile_difference,p95_absolute_percentile_difference
0,Overall,11960,0.9992,1.3195,0.9883,3.5584
1,DF,5065,0.9996,0.7861,0.6550,1.9296
2,MF,4105,0.9994,1.5463,1.3286,3.6039
3,FW,2790,0.9992,1.9539,1.8452,4.2892


## 2. Validation 1 — Distribution
The index is mean-zero within each season x role cell by construction. Its shape (skew, tails, outliers) is the first check of whether it behaves sensibly.

In [6]:
def summarize(s):
    return pd.Series({
        'n': s.count(), 'mean': s.mean(), 'sd': s.std(),
        'p5': s.quantile(.05), 'median': s.median(), 'p95': s.quantile(.95), 'max': s.max(),
        'skew': stats.skew(s.dropna()), 'kurtosis_excess': stats.kurtosis(s.dropna()),
    })

dist_by_role = primary.groupby('role', observed=True)['role_index_z'].apply(summarize).unstack()
dist_by_role


,n,mean,sd,p5,median,p95,max,skew,kurtosis_excess
role,,,,,,,,,
DF,6069.0,-8.078341e-17,0.514797,-0.799121,-0.032331,0.897583,2.216595,0.350578,0.277170
FW,3574.0,-4.771412e-17,0.670749,-0.912274,-0.100016,1.234074,4.132193,0.966835,1.950383
MF,5018.0,-1.274389e-17,0.622522,-0.879148,-0.082123,1.118148,3.973016,0.837117,1.482241


In [7]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, role in zip(axes, ['DF','MF','FW']):
    d = primary[primary['role']==role]['role_index_z']
    ax.hist(d, bins=50, color='#2b6ca3', edgecolor='white', linewidth=0.3)
    ax.set_title(f"{role} (n={len(d)})"); ax.set_xlabel("Role-specific index (z)"); ax.axvline(0, color='k', lw=0.8)
plt.tight_layout(); plt.savefig(P["figures"] / "02_target_distribution_by_role.png", dpi=150, bbox_inches="tight"); plt.show()


/tmp/ipykernel_4093/2374952155.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(P["figures"] / "02_target_distribution_by_role.png", dpi=150, bbox_inches="tight"); plt.show()


In [8]:
outliers = primary[primary['role_index_z'].abs() > 3][['player','season','role','role_index_z','Playing Time_Min']].sort_values('role_index_z')
print(f"Player-seasons with |z| > 3: {len(outliers)}")
outliers


Player-seasons with |z| > 3: 11


,player,season,role,role_index_z,Playing Time_Min
11676,Lionel Messi,2223,FW,3.054181,2837
11673,Lionel Messi,1920,FW,3.074200,2880
12179,Luis Muriel,2021,FW,3.081292,1436
15047,Neymar,1819,MF,3.286581,1439
11221,Kylian Mbappé,1920,FW,3.344148,1513
15765,Ousmane Dembélé,2425,FW,3.431407,1730
11671,Lionel Messi,1718,FW,3.632280,3000
11672,Lionel Messi,1819,FW,3.797638,2713
15049,Neymar,2021,MF,3.866451,1415
15048,Neymar,1920,MF,3.973016,1318


**Finding:** all three role distributions are right-skewed (expected — most players cluster near average, a few elite performers create a long tail). The 11 |z|>3 outliers are all elite attacking talent (Messi x4, Neymar x4, Mbappe, Dembele, Muriel) — strong informal face validity.

## 3. Validation 2 — Top- and Bottom-Ranked Players by Role
Pooled across all eight seasons (>=450 minutes).

In [9]:
for role in ['FW','MF','DF']:
    d = primary[primary['role']==role].sort_values('role_index_z', ascending=False)
    print(f"\n=== {role} TOP 10 ===")
    display(d.head(10)[['player','season','team_primary','role_index_z']])
    print(f"=== {role} BOTTOM 10 ===")
    display(d.tail(10)[['player','season','team_primary','role_index_z']])



=== FW TOP 10 ===


,player,season,team_primary,role_index_z
15046,Neymar,1718,Paris S-G,4.132193
11672,Lionel Messi,1819,Barcelona,3.797638
11671,Lionel Messi,1718,Barcelona,3.632280
15765,Ousmane Dembélé,2425,Paris S-G,3.431407
11221,Kylian Mbappé,1920,Paris S-G,3.344148
12179,Luis Muriel,2021,Atalanta,3.081292
11673,Lionel Messi,1920,Barcelona,3.074200
11676,Lionel Messi,2223,Paris S-G,3.054181
11674,Lionel Messi,2021,Barcelona,2.896685
11675,Lionel Messi,2122,Paris S-G,2.719259


=== FW BOTTOM 10 ===


,player,season,team_primary,role_index_z
6096,Felipe Avenatti,1718,Bologna,-1.335435
20622,Yacine Bammou,1819,Caen,-1.336582
8627,Jan Thielmann,1920,Köln,-1.369939
18432,Sergio León,1819,Betis,-1.375128
5301,Elias Kachunga,1819,Huddersfield,-1.420901
16490,Pietro Pellegri,2324,Torino,-1.438899
14649,Moussa Konaté,1920,Amiens,-1.446619
20462,William Osula,2324,Sheffield Utd,-1.490965
18538,Shon Weissman,2324,Salernitana,-1.526302
14668,Moussa Soumano,2223,Ajaccio,-1.566736



=== MF TOP 10 ===


...

,player,season,team_primary,role_index_z
18197,Sebastian Griesbeck,2021,Union Berlin,-1.325981
16926,Reinhold Yabo,2021,Arminia,-1.330269
13485,Mathias Olesen,2223,Köln,-1.333071
9833,Joseph Nduquidi,2324,Metz,-1.343491
2262,Baissama Sankoh,1819,Caen,-1.350810
20630,Yan Brice Eteki,2021,Granada,-1.354212
13510,Mathieu Flamini,1718,Getafe,-1.379781
11685,Ljubomir Fejsa,1920,Alavés,-1.388022
13511,Mathieu Flamini,1819,Getafe,-1.421102
8198,Ivan Radovanović,2223,Salernitana,-1.423469



=== DF TOP 10 ===


,player,season,team_primary,role_index_z
18994,Strahinja Pavlović,2122,Monaco,2.216595
13566,Mats Wieffer,2425,Brighton,2.014486
13967,Maximilian Mittelstädt,2324,Stuttgart,1.982103
1644,Anthony Caci,1920,Strasbourg,1.947179
859,Alexander Hack,1920,Mainz 05,1.925427
5459,Emiliano Velázquez,1819,Rayo Vallecano,1.921291
10045,João Cancelo,2122,Manchester City,1.886121
17020,Ricardo Pereira,1920,Leicester City,1.880426
858,Alexander Hack,1819,Mainz 05,1.835224
20857,Youcef Atal,2223,Nice,1.832858


=== DF BOTTOM 10 ===


,player,season,team_primary,role_index_z
1887,Archie Gray,2425,Tottenham,-1.279378
3660,Conor Coady,1819,Wolves,-1.280109
11686,Lloyd Kelly,1920,Bournemouth,-1.296178
17051,Riccardo Marchizza,2324,Frosinone,-1.298871
13248,Martin Olsson,1718,Swansea City,-1.313283
2281,Bamo Meïté,2425,Montpellier,-1.387192
12956,Marcus Pedersen,2425,Torino,-1.405106
1770,Antonio Barreca,1920,Genoa,-1.590785
17106,Ridgeciano Haps,2324,Genoa,-1.627843
21140,Álvaro González,2122,Marseille,-1.992991


In [10]:
# Spot-check some less-famous top DF names against their underlying components, to rule out small-sample artefacts
cols = ['player','season','team_primary','Playing Time_Min','role_index_z','z_attack','z_creation','z_possession','z_defense']
display(primary[primary['player'].isin(['Paul Joly','Ismaily','Mats Wieffer'])][cols])


,player,season,team_primary,Playing Time_Min,role_index_z,z_attack,z_creation,z_possession,z_defense
8108,Ismaily,2223,Lille,1779,0.300102,0.804404,0.951412,0.603399,-0.147800
8109,Ismaily,2324,Lille,2465,0.417240,-0.701867,0.166738,0.781485,0.534090
8110,Ismaily,2425,Lille,904,1.346864,0.812086,0.926282,1.405618,1.550617
13566,Mats Wieffer,2425,Brighton,1015,2.014486,1.351124,1.611299,1.138925,2.705895
16100,Paul Joly,2223,Auxerre,867,1.310341,-0.841151,0.584956,1.525279,1.850785
16101,Paul Joly,2425,Auxerre,1009,1.584276,0.807544,1.623159,0.482578,2.278807


**Finding:** FW/MF top lists are dominated by widely recognised elite attackers — strong face validity. The DF top list is more heterogeneous, blending attack-minded full-backs (Cancelo, Alexander-Arnold, Hakimi) with some less prominent names; spot-checks confirm these are driven by genuinely elite defensive/progression output at substantive minutes, not small-sample noise. This reveals that the broad 'DF' bucket mixes stylistically different sub-roles — a limitation carried into Section 6.

## 4. Validation 3 — Stability Across Leagues and Seasons

### 4.1 Across leagues
Because standardisation is pooled across leagues, a league effect here is a genuine stylistic difference, not a computational artefact.

In [11]:
league_tab = primary.groupby(['role','league_primary'], observed=True)['role_index_z'].agg(['count','mean','std']).round(3)
league_tab


count   mean    std
role league_primary                         
DF   ENG-Premier League   1198  0.080  0.519
     ESP-La Liga          1244 -0.142  0.464
     FRA-Ligue 1          1194  0.032  0.533
     GER-Bundesliga       1106  0.159  0.522
     ITA-Serie A          1327 -0.100  0.478
FW   ENG-Premier League    740  0.077  0.638
     ESP-La Liga           739 -0.091  0.671
     FRA-Ligue 1           726 -0.007  0.715
     GER-Bundesliga        636  0.075  0.660
     ITA-Serie A           733 -0.046  0.652
MF   ENG-Premier League    996  0.042  0.627
     ESP-La Liga          1120 -0.072  0.614
     FRA-Ligue 1           978  0.075  0.631
     GER-Bundesliga        891 -0.029  0.635
     ITA-Serie A          1033 -0.008  0.598

In [12]:
anova_rows = []
for role in ['DF','MF','FW']:
    d = primary[primary['role']==role]
    groups = [g['role_index_z'].dropna().values for _, g in d.groupby('league_primary', observed=True)]
    f, p = stats.f_oneway(*groups)
    eta_sq = (f*(len(groups)-1)) / (f*(len(groups)-1) + (len(d)-len(groups)))
    anova_rows.append({'role':role,'F':round(f,2),'p':p,'approx_eta_sq':round(eta_sq,4)})
pd.DataFrame(anova_rows)


,role,F,p,approx_eta_sq
0,DF,74.23,1.598658e-61,0.0467
1,MF,9.03,2.883552e-07,0.0072
2,FW,8.79,4.668269e-07,0.0098


In [13]:
fig, ax = plt.subplots(figsize=(8,4.5))
leagues = sorted(league_tab.index.get_level_values('league_primary').unique())
x = np.arange(len(leagues)); width = 0.25
colors = {'DF':'#c0392b','MF':'#2b6ca3','FW':'#27ae60'}
for i, role in enumerate(['DF','MF','FW']):
    vals = [league_tab.loc[(role,l),'mean'] for l in leagues]
    ax.bar(x+i*width, vals, width, label=role, color=colors[role])
ax.set_xticks(x+width); ax.set_xticklabels([l.split('-')[-1] for l in leagues], rotation=20)
ax.axhline(0, color='k', lw=0.8); ax.set_ylabel("Mean index (z)"); ax.legend()
plt.tight_layout(); plt.savefig(P["figures"] / "02_target_league_means.png", dpi=150, bbox_inches="tight"); plt.show()


/tmp/ipykernel_4093/414356643.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(P["figures"] / "02_target_league_means.png", dpi=150, bbox_inches="tight"); plt.show()


**Finding:** league effect is statistically significant everywhere (large n) but practically small except for DF (eta^2 ~4.7% vs <1% for MF/FW) — La Liga/Serie A defenders score systematically lower than Bundesliga/Premier League defenders, plausibly a stylistic effect.

### 4.2 Across seasons
Season-to-season Spearman rank correlation for players retained across each consecutive season pair.

In [14]:
rows = []
for role in ['DF','MF','FW']:
    d = primary[primary['role']==role][['player_id','season','role_index_z']].copy()
    d['season'] = d['season'].astype(str)
    for i in range(len(SEASON_ORDER)-1):
        s0, s1 = SEASON_ORDER[i], SEASON_ORDER[i+1]
        a = d[d['season']==s0].set_index('player_id')['role_index_z']
        b = d[d['season']==s1].set_index('player_id')['role_index_z']
        common = a.index.intersection(b.index)
        if len(common) >= 10:
            rho, p = stats.spearmanr(a.loc[common], b.loc[common])
            rows.append({'role':role,'season_t':s0,'season_t1':s1,'n_retained':len(common),'spearman_rho':round(rho,3)})
season_stab = pd.DataFrame(rows)
print(season_stab.groupby('role')['spearman_rho'].mean())
season_stab


role
DF    0.590286
FW    0.603000
MF    0.729714
Name: spearman_rho, dtype: float64


,role,season_t,season_t1,n_retained,spearman_rho
0,DF,1718,1819,530,0.625
1,DF,1819,1920,515,0.640
2,DF,1920,2021,530,0.613
3,DF,2021,2122,529,0.607
4,DF,2122,2223,535,0.529
5,DF,2223,2324,541,0.562
6,DF,2324,2425,516,0.556
7,MF,1718,1819,405,0.717
8,MF,1819,1920,402,0.728
9,MF,1920,2021,407,0.768


**Finding:** moderate-strong year-on-year persistence (DF ~0.59, MF ~0.73, FW ~0.60 average rho), consistent with the literature's finding that lagged performance is a strong autoregressive predictor — supports using the index as a meaningful forecasting target rather than single-season noise.

## 5. Validation 4 — Sensitivity to Component Weights
Two checks: (a) deliberately different named weighting philosophies, and (b) Monte Carlo perturbation of the primary weights (+/-0.15 additive noise, renormalised, 200 draws per role).

In [15]:
ALT_SCHEMES = {
    'equal_weights': {r: {c:0.25 for c in COMPONENTS} for r in ['FW','MF','DF']},
    'role_agnostic_MFweights': {r: PRIMARY_W['MF'] for r in ['FW','MF','DF']},
    'defense_deemphasised_DF': {**PRIMARY_W, 'DF': {'attack':0.15,'creation':0.20,'possession':0.30,'defense':0.35}},
    'attack_heavy_all': {r: {'attack':0.55,'creation':0.20,'possession':0.15,'defense':0.10} for r in ['FW','MF','DF']},
}

elig = df[df['eligible_450'] & df['role'].isin(['DF','MF','FW'])].copy()
raw_cols = [f'raw_{c}' for c in COMPONENTS]
g = elig.groupby(['season','role'], observed=True)[raw_cols]
z = (elig[raw_cols] - g.transform('mean')) / g.transform('std')
z.columns = [f'z_{c}' for c in COMPONENTS]
elig = pd.concat([elig, z], axis=1)

def composite(elig, weights):
    idx = np.zeros(len(elig))
    for role, w in weights.items():
        mask = (elig['role']==role).values
        role_z = np.column_stack([elig.loc[mask, f'z_{c}'].values for c in COMPONENTS])
        role_w = np.array([w[c] for c in COMPONENTS])
        idx[mask] = role_z @ role_w
    return idx

elig['idx_primary'] = composite(elig, PRIMARY_W)
results = []
for name, scheme in ALT_SCHEMES.items():
    elig[f'idx_{name}'] = composite(elig, scheme)
    for role in ['DF','MF','FW']:
        d = elig[elig['role']==role]
        rho, _ = stats.spearmanr(d['idx_primary'], d[f'idx_{name}'])
        top_n = max(20, int(0.02*len(d)))
        top_primary = set(d.nlargest(top_n, 'idx_primary')['player_id']+d.nlargest(top_n,'idx_primary')['season'].astype(str))
        top_alt = set(d.nlargest(top_n, f'idx_{name}')['player_id']+d.nlargest(top_n,f'idx_{name}')['season'].astype(str))
        jaccard = len(top_primary & top_alt) / len(top_primary | top_alt)
        results.append({'scheme':name,'role':role,'spearman_vs_primary':round(rho,3),'top20_jaccard':round(jaccard,3)})
pd.DataFrame(results)


,scheme,role,spearman_vs_primary,top20_jaccard
0,equal_weights,DF,0.744,0.192
1,equal_weights,MF,0.988,0.852
2,equal_weights,FW,0.858,0.511
3,role_agnostic_MFweights,DF,0.659,0.192
4,role_agnostic_MFweights,MF,1.000,1.000
5,role_agnostic_MFweights,FW,0.835,0.449
6,defense_deemphasised_DF,DF,0.895,0.367
7,defense_deemphasised_DF,MF,1.000,1.000
8,defense_deemphasised_DF,FW,1.000,1.000
9,attack_heavy_all,DF,0.506,0.095


In [16]:
rng = np.random.default_rng(42)
mc_rows = []
for role in ['DF','MF','FW']:
    base = np.array([PRIMARY_W[role][c] for c in COMPONENTS])
    d = elig[elig['role']==role]
    rhos = []
    for _ in range(200):
        noise = rng.uniform(-0.15, 0.15, size=4)
        w = np.clip(base + noise, 0.01, None); w = w / w.sum()
        alt_idx = (d[[f'z_{c}' for c in COMPONENTS]].values @ w)
        rho, _ = stats.spearmanr(d['idx_primary'], alt_idx)
        rhos.append(rho)
    mc_rows.append({'role':role, 'mc_rho_mean':round(np.mean(rhos),3), 'mc_rho_p5':round(np.percentile(rhos,5),3), 'mc_rho_p95':round(np.percentile(rhos,95),3)})
pd.DataFrame(mc_rows)


,role,mc_rho_mean,mc_rho_p5,mc_rho_p95
0,DF,0.959,0.885,0.996
1,MF,0.975,0.930,0.997
2,FW,0.976,0.936,0.997


**Finding:** under realistic weight uncertainty (Monte Carlo), rankings are highly stable everywhere (mean rho >= 0.96). Under deliberately different philosophies, MF/FW stay reasonably robust (rho 0.83-1.00) but **DF rankings are markedly more weight-sensitive** (rho as low as 0.51, top-20 overlap as low as 9.5% under an attack-heavy scheme) — the same DF-instability pattern seen in Sections 3-4.

## 6. Validation boundary — external cross-source comparison excluded

An external cross-source comparison would require six independently obtained `.rds` files that are not part of the supplied project materials and cannot be regenerated from the Kaggle source dataset used here. That comparison is therefore excluded from the reported evidence. The target-validation results retained in this notebook are reproducible from `Dataset_Original.csv` and the preceding player-season aggregation.

## 7. Validation 6 — Role and Attacking-Statistic Dominance
Because the four components are correlated with each other, a component's *nominal* weight is not the same as its *realised* contribution to the index's variance. This decomposes each role's index variance into the share attributable to each component: `contribution_i = weight_i x cov(z_i, index) / var(index)` (shares sum to 100% within a role).

In [17]:
rows = []
for role in ['DF','MF','FW']:
    d = primary[primary['role']==role]
    idx = d['role_index_z'].values
    var_idx = np.var(idx, ddof=1)
    w = PRIMARY_W[role]
    for c in COMPONENTS:
        zc = d[f'z_{c}'].values
        cov = np.cov(zc, idx, ddof=1)[0,1]
        contrib = w[c]*cov/var_idx
        r_simple, _ = stats.pearsonr(zc, idx)
        rows.append({'role':role,'component':c,'weight':w[c],'variance_share_pct':round(contrib*100,1),'corr_with_index':round(r_simple,3)})
var_decomp = pd.DataFrame(rows)
var_decomp


,role,component,weight,variance_share_pct,corr_with_index
0,DF,attack,0.10,5.8,0.300
1,DF,creation,0.15,11.2,0.385
2,DF,possession,0.25,22.7,0.467
3,DF,defense,0.50,60.3,0.621
4,MF,attack,0.20,21.0,0.654
5,MF,creation,0.35,51.8,0.922
6,MF,possession,0.25,33.8,0.842
7,MF,defense,0.20,-6.6,-0.206
8,FW,attack,0.50,61.6,0.828
9,FW,creation,0.25,26.8,0.719


In [18]:
fig, ax = plt.subplots(figsize=(7,4.5))
comp_colors = {'attack':'#e74c3c','creation':'#3498db','possession':'#2ecc71','defense':'#9b59b6'}
bottom = np.zeros(3); roles=['DF','MF','FW']
for c in COMPONENTS:
    vals = [var_decomp[(var_decomp.role==r)&(var_decomp.component==c)]['variance_share_pct'].values[0] for r in roles]
    ax.bar(roles, vals, bottom=bottom, label=c, color=comp_colors[c]); bottom += np.array(vals)
ax.axhline(0, color='k', lw=0.8); ax.set_ylabel("Share of index variance (%)"); ax.legend()
plt.tight_layout(); plt.savefig(P["figures"] / "02_target_variance_decomposition.png", dpi=150, bbox_inches="tight"); plt.show()


/tmp/ipykernel_4093/2770710388.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(P["figures"] / "02_target_variance_decomposition.png", dpi=150, bbox_inches="tight"); plt.show()


In [19]:
df['ga_per90'] = df['Performance_G+A']/df['90s_']
for role in ['DF','MF','FW']:
    merged_ga = primary[primary['role']==role].merge(df[['player_id','season','ga_per90']], on=['player_id','season'], how='left')
    r,_ = stats.pearsonr(merged_ga['ga_per90'].fillna(0), merged_ga['role_index_z'])
    print(role, 'corr(index, raw G+A per90) =', round(r,3))


DF corr(index, raw G+A per90) = 0.213
MF corr(index, raw G+A per90) = 0.568
FW corr(index, raw G+A per90) = 0.642


**Finding — the key limitation of this index:** for FW, attack appropriately dominates (61.6% of variance vs. 50% nominal weight) — design works as intended. For DF, defense similarly dominates appropriately (60.3% vs 50%). **For MF, the defense component's realised contribution is *negative* (~-6.6%) despite a positive 20% nominal weight** — because defensive and creative output are negatively correlated among midfielders (r ~ -0.49 to -0.51): the highest creators tend to defend the least, so a positively-weighted defense term actually *penalises* the top creators. Raw attacking output (G+A per90) correlates with the composite even outside forwards (MF r~0.57, DF r~0.21), confirming attacking production pulls the ranking beyond its nominal weight, most acutely in midfield. **In practice, the current MF index behaves closer to a creative-attacking index than a balanced two-way midfielder rating** — flagged as the top candidate for refinement (e.g. orthogonalising components, or reweighting).

## 8. Save the final index dataset

In [20]:
full = df[['player_id','player','season','season_label','team_primary','league_primary','role','pos_','age_',
           'Playing Time_Min','eligible_450','eligible_900','raw_attack','raw_creation','raw_possession','raw_defense']].copy()

p450 = primary[['player_id','season','role_index_z','role_index_pct','group_n']].rename(
    columns={'role_index_z':'index_450min','role_index_pct':'index_450min_pctile','group_n':'group_n_450min'})
p900 = primary900[['player_id','season','role_index_z','role_index_pct']].rename(
    columns={'role_index_z':'index_900min_sensitivity','role_index_pct':'index_900min_pctile'})

full = full.merge(p450, on=['player_id','season'], how='left').merge(p900, on=['player_id','season'], how='left')
full.to_csv(P["processed"] / "Role_Specific_Index_Dataset.csv", index=False)
print("Saved Role_Specific_Index_Dataset.csv:", full.shape)
full[full['role'].isin(['DF','MF','FW'])].dropna(subset=['index_450min']).shape


Saved Role_Specific_Index_Dataset.csv: (21371, 21)


(14661, 21)

## 8. Canonical exports

The final target dataset and report-ready validation tables are written to the package output folders. These files are regenerated from the player-season panel whenever the frozen pipeline is rerun.


In [21]:
# Export report-ready target-validation tables
dist_by_role.to_csv(P["tables"] / "02_target_distribution_by_role.csv")
season_stab.to_csv(P["tables"] / "02_target_season_stability.csv", index=False)
pd.DataFrame(mc_rows).to_csv(P["tables"] / "02_target_weight_sensitivity_monte_carlo.csv", index=False)
var_decomp.to_csv(P["tables"] / "02_target_variance_decomposition.csv", index=False)
print("Saved target-validation tables to", P["tables"])
league_tab.reset_index().to_csv(P["tables"] / "02_target_league_means.csv", index=False)
target_900_sensitivity.to_csv(P["tables"] / "02_target_900min_sensitivity.csv", index=False)


Saved target-validation tables to /mnt/data/CSMPR_Frozen_Analysis_Final/results/tables


## Summary

The role-specific target used by the forecasting analysis is defined by:

- broad roles DF/MF/FW from the first-listed position code; goalkeepers excluded;
- a primary eligibility threshold of at least 450 minutes;
- attack, creation, possession and defence components expressed per 90 and standardised within season × role across the five leagues;
- fixed role-specific component weights established before predictive modelling;
- a 0–100 percentile rank used as the next-season prediction target;
- sensitivity analyses covering temporal persistence, alternative component weights, league variation and realised component contribution;
- a 900-minute **target-level** sensitivity reference that tests the effect of a stricter eligibility population without rerunning the forecasting models.

The 900-minute comparison is intentionally limited to target construction. A future extension could repeat the full chronological modelling analysis at a stricter minutes threshold to examine the trade-off between a more established-player population and reduced sample coverage.